# 🛡️ Project 10: Phishing URL & Malicious Domain Detection
### Cybersecurity Machine Learning, Shannon Entropy & High-Specificity Tuning

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟡 Intermediate  
**Domain:** Cybersecurity & Trust & Safety  

---
### Notebook Outline:
1. **Environment Setup**
2. **URL Dataset Ingestion & Class Balance Inspection**
3. **Exploratory Data Analysis: Shannon Entropy & Subdomain Depth**
4. **Lexical Feature Engineering**
5. **Model Training: LightGBM with High Specificity Constraints**
6. **Operating Point Selection for Zero-False-Alarm Deployment**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Cybersecurity workspace configured.")

In [ ]:
# Ingestion
df = pd.read_csv("data/phishing_urls.csv")
print(f"Inspected URLs: {len(df)} | Phishing Prevalence: {df['is_phishing'].mean():.2%}")
display(df.head(4))

In [ ]:
# Modeling & High-Specificity Operating Point Selection
features = ['url_length', 'subdomain_count', 'shannon_entropy', 'has_ip_in_host', 'has_https', 'suspicious_keyword_count']
X = df[features]
y = df['is_phishing']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = lgb.LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.04, random_state=42, verbose=-1)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")

# Find threshold ensuring Specificity >= 99.5%
neg_mask = (y_test == 0)
threshold_995 = np.percentile(probs[neg_mask], 99.5)
preds_strict = (probs >= threshold_995).astype(int)

cm = confusion_matrix(y_test, preds_strict)
print(f"Strict Operational Threshold: {threshold_995:.4f}")
print(f"False Positives (Benign URLs blocked): {cm[0, 1]} / {neg_mask.sum()} (Specificity: {(cm[0,0]/neg_mask.sum()):.2%})")
print(f"Recall on Phishing Attacks: {(cm[1, 1] / (y_test == 1).sum()):.2%}")